In [0]:
%sql
CREATE OR REPLACE TABLE workspace.finance.gold_monthly_summary AS
SELECT
  CAST(txn_month AS DATE) AS month,
  sum(CASE WHEN direction = 'CREDIT' AND category IN ('Salary', 'Other Income') THEN amount ELSE 0 END) AS income,
  sum(CASE WHEN direction = 'DEBIT'  AND category <> 'Transfer'                 THEN amount ELSE 0 END) AS gross_spend,
  sum(CASE WHEN direction = 'CREDIT' AND category = 'Refund'                    THEN amount ELSE 0 END) AS refunds
FROM workspace.finance.transactions_categorized
GROUP BY 1;

-- derived columns, added with a simple follow-up statement
CREATE OR REPLACE TABLE workspace.finance.gold_monthly_summary AS
SELECT *,
  gross_spend - refunds                                   AS net_spend,
  income - (gross_spend - refunds)                        AS savings,
  round((income - (gross_spend - refunds)) / income, 3)   AS savings_rate
FROM workspace.finance.gold_monthly_summary;

SELECT * FROM workspace.finance.gold_monthly_summary ORDER BY month;

month,income,gross_spend,refunds,net_spend,savings,savings_rate
2026-04-01,95000.00,46254.25,0.00,46254.25,48745.75,0.513
2026-05-01,95000.00,56745.53,0.00,56745.53,38254.47,0.403
2026-06-01,95000.00,67622.83,0.00,67622.83,27377.17,0.288
2026-07-01,95000.00,53475.48,0.00,53475.48,41524.52,0.437
2026-08-01,95000.00,121167.57,1250.00,119917.57,-24917.57,-0.262
2026-09-01,95000.00,48538.11,0.00,48538.11,46461.89,0.489


In [0]:
%sql
CREATE OR REPLACE VIEW workspace.finance.gold_category_spend AS
WITH monthly AS (
  SELECT CAST(txn_month AS DATE) AS month, category, sum(amount) AS spent, count(*) AS txns
  FROM workspace.finance.transactions_categorized
  WHERE direction = 'DEBIT' AND category <> 'Transfer'
  GROUP BY 1, 2
)
SELECT *,
  LAG(spent) OVER (PARTITION BY category ORDER BY month)          AS prev_month_spent,
  round(100 * (spent - LAG(spent) OVER (PARTITION BY category ORDER BY month))
        / LAG(spent) OVER (PARTITION BY category ORDER BY month), 1) AS change_pct
FROM monthly;

SELECT * FROM workspace.finance.gold_category_spend
WHERE category = 'Food & Dining' ORDER BY month;

month,category,spent,txns,prev_month_spent,change_pct
2026-04-01,Food & Dining,5887.69,17,null,null
2026-05-01,Food & Dining,9551.37,14,5887.69,62.2
2026-06-01,Food & Dining,11652.80,17,9551.37,22.0
2026-07-01,Food & Dining,7793.96,20,11652.80,-33.1
2026-08-01,Food & Dining,11445.19,14,7793.96,46.8
2026-09-01,Food & Dining,6092.79,11,11445.19,-46.8


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.finance.budgets (category STRING, monthly_budget DECIMAL(12,2));

INSERT INTO workspace.finance.budgets VALUES
  ('Rent', 18500), ('Food & Dining', 8000), ('Groceries', 5000), ('Transport', 3500),
  ('Shopping', 15000), ('Entertainment', 1500), ('Utilities', 3500), ('Health', 4000), ('Travel', 10000);

  CREATE OR REPLACE VIEW workspace.finance.gold_budget_status AS
SELECT
  s.month, s.category, b.monthly_budget, s.spent,
  b.monthly_budget - s.spent               AS remaining,
  round(s.spent / b.monthly_budget, 3)     AS pct_used,
  CASE WHEN s.spent > b.monthly_budget        THEN 'OVER'
       WHEN s.spent >= 0.8 * b.monthly_budget THEN 'WARNING'
       ELSE 'OK' END                        AS status
FROM workspace.finance.gold_category_spend s
JOIN workspace.finance.budgets b USING (category);

SELECT * FROM workspace.finance.gold_budget_status
WHERE status <> 'OK' ORDER BY month, pct_used DESC;

month,category,monthly_budget,spent,remaining,pct_used,status
2026-04-01,Transport,3500.00,4016.59,-516.59,1.148,OVER
2026-04-01,Rent,18500.00,18000.00,500.00,0.973,WARNING
2026-04-01,Groceries,5000.00,4716.01,283.99,0.943,WARNING
2026-05-01,Food & Dining,8000.00,9551.37,-1551.37,1.194,OVER
2026-05-01,Health,4000.00,4497.00,-497.00,1.124,OVER
2026-05-01,Transport,3500.00,3590.96,-90.96,1.026,OVER
2026-05-01,Rent,18500.00,18000.00,500.00,0.973,WARNING
2026-05-01,Shopping,15000.00,13919.53,1080.47,0.928,WARNING
2026-05-01,Groceries,5000.00,4063.95,936.05,0.813,WARNING
2026-06-01,Food & Dining,8000.00,11652.80,-3652.80,1.457,OVER


In [0]:
%sql
CREATE OR REPLACE VIEW workspace.finance.gold_top_merchants AS
SELECT * FROM (
  SELECT
    CAST(txn_month AS DATE) AS month, merchant, category,
    sum(amount) AS spent, count(*) AS txns,
    RANK() OVER (PARTITION BY txn_month ORDER BY sum(amount) DESC) AS rank
  FROM workspace.finance.transactions_categorized
  WHERE direction = 'DEBIT' AND category <> 'Transfer'
  GROUP BY txn_month, merchant, category
)
WHERE rank <= 5;

SELECT * FROM workspace.finance.gold_top_merchants ORDER BY month, rank;

month,merchant,category,spent,txns,rank
2026-04-01,Rent,Rent,18000.00,1,1
2026-04-01,Flipkart,Shopping,4928.74,2,2
2026-04-01,Fuel,Transport,3429.01,3,3
2026-04-01,BigBasket,Groceries,3182.01,4,4
2026-04-01,Swiggy,Food & Dining,2270.57,6,5
2026-05-01,Rent,Rent,18000.00,1,1
2026-05-01,Flipkart,Shopping,6699.52,2,2
2026-05-01,Barbeque Nation,Food & Dining,4704.96,2,3
2026-05-01,Cult.fit,Health,4497.00,3,4
2026-05-01,Amazon,Shopping,3991.27,4,5


In [0]:
%sql
CREATE OR REPLACE VIEW workspace.finance.gold_unusual_transactions AS
SELECT txn_date, merchant, category, amount,
       round(cat_avg, 2) AS category_average,
       round(amount / cat_avg, 1) AS times_average,
       description
FROM (
  SELECT *,
    avg(amount)    OVER (PARTITION BY category) AS cat_avg,
    stddev(amount) OVER (PARTITION BY category) AS cat_std,
    count(*)       OVER (PARTITION BY category) AS cat_n
  FROM workspace.finance.transactions_categorized
  WHERE direction = 'DEBIT' AND category <> 'Transfer'
)
WHERE cat_n >= 5                          -- enough history to judge
  AND amount > cat_avg + 3 * cat_std      -- statistically unusual
  AND amount >= 5000;                     -- and big enough to matter

SELECT * FROM workspace.finance.gold_unusual_transactions;

txn_date,merchant,category,amount,category_average,times_average,description
2026-08-13,Croma,Shopping,64990.00,3395.05,19.1,POS 453212******1234 CROMA ELECTRONICS
